# Tinjauan transform

Satu bagian per langkah transform. Di setiap bagian:
1. **Kode**: isi fungsi dari berkas `.py` (bukan salinan, jadi selalu yang terbaru).
2. **Hasil**: fungsi itu dijalankan pada data asli di `data_lake`.

Jalankan sel **Persiapan** dulu, lalu bagian yang ingin ditinjau.

In [28]:
# Persiapan
import glob
import importlib
import inspect
import json
import os
import sys
from pathlib import Path

# notebook ini ada di data-science/transform/; kerja dari data-science/
# supaya `from transform import ...` dan path data_lake/ jalan
akar = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "transform" / "parser.py").exists())
os.chdir(akar)
if str(akar) not in sys.path:
    sys.path.insert(0, str(akar))

import polars as pl
from IPython.display import HTML, display
from pygments import highlight
from pygments.formatters import HtmlFormatter
from pygments.lexers import PythonLexer

from transform import parser

importlib.reload(parser)  # ambil versi .py terbaru tanpa restart kernel
pl.Config.set_tbl_rows(-1)
pl.Config.set_fmt_str_lengths(90)


def tampilkan_kode(objek):
    """Kode sumber modul/fungsi dengan warna tema gelap (github-dark)."""
    gaya = HtmlFormatter(style="github-dark", noclasses=True, nobackground=True)
    sumber = objek if isinstance(objek, str) else inspect.getsource(objek)
    isi = highlight(sumber, PythonLexer(), gaya)
    display(HTML(f'<div style="background:#0d1117;padding:12px;border-radius:6px">{isi}</div>'))

## Task 1A: berat (`transform/parser.py`)

Aturan: spec bagian 5.2. Fungsi yang dipakai transform nanti:
- `berat_gram(teks)`: berat dalam gram dari nama / variant / slug, atau `None`.
- `satuan_baris(nama, variant, banyak_opsi, usp)`: berat + sumber + jenis satuan + jumlah untuk satu baris. Produk **beberapa opsi** hanya membaca `variant` (A1).
- `pilih_berat(kandidat)`: sumber pertama yang punya berat (Blibli: `variant` → `nama`).
- `berat_dari_slug(url)`: cadangan Tokopedia kalau nama tanpa berat.
- `mangga_manalagi(nama)`: memuat mangga (atau ejaan lain: manga, mango, m4ngg4, …) **dan** manalagi.
- `jumlah_buah(nama)`, `jenis_satuan(nama, berat)`: `berat` / `jumlah` / `kemasan` / `lain`.

### Kode

In [29]:
tampilkan_kode(parser)

### Hasil: semua nama Mangga Manalagi unik (Blibli + Tokopedia, semua tanggal)

`cadangan` = `variant` (Blibli) atau URL tanpa query (Tokopedia).

In [30]:
baris = []
for berkas in glob.glob("data_lake/raw/blibli/date=*/*_page_*.json"):
    data = json.load(open(berkas, encoding="utf-8")).get("data") or {}
    for p in data.get("products") or []:
        if parser.mangga_manalagi(p.get("name")):
            baris.append({"sumber": "blibli", "nama": p["name"], "cadangan": p.get("variant") or "",
                           "banyak_opsi": bool(p.get("hasVariants")), "usp": p.get("uniqueSellingPoint")})
for berkas in glob.glob("data_lake*/raw/tokopedia/date=*/page_*.json"):
    try:
        produk = json.load(open(berkas, encoding="utf-8"))[0]["data"]["searchProductV5"]["data"]["products"]
    except (ValueError, KeyError, IndexError, TypeError):
        continue
    for p in produk:
        if parser.mangga_manalagi(p.get("name")):
            baris.append({"sumber": "tokopedia", "nama": p["name"], "cadangan": (p.get("url") or "").split("?")[0],
                           "banyak_opsi": False, "usp": None})


def hitung(b):
    if b["sumber"] == "blibli":
        s = parser.satuan_baris(b["nama"], b["cadangan"], b["banyak_opsi"], b["usp"])
        berat, sumber_berat, jenis = s.berat_gram, s.sumber_berat, s.jenis_satuan
    else:
        berat, sumber_berat = parser.pilih_berat([("nama", b["nama"])])
        if berat is None:
            berat = parser.berat_dari_slug(b["cadangan"])
            sumber_berat = "url" if berat is not None else None
        jenis = parser.jenis_satuan(b["nama"], berat)
    return {**{k: v for k, v in b.items() if k != "usp"},
            "berat_gram": berat, "sumber_berat": sumber_berat, "jenis_satuan": jenis}


berat = pl.DataFrame([hitung(b) for b in baris]).unique().sort("sumber", "jenis_satuan", "nama")
display(berat.group_by("sumber", "jenis_satuan").len().sort("sumber", "jenis_satuan"))
berat

sumber,jenis_satuan,len
str,str,u32
"""blibli""","""berat""",8
"""tokopedia""","""berat""",34
"""tokopedia""","""lain""",7


sumber,nama,cadangan,banyak_opsi,berat_gram,sumber_berat,jenis_satuan
str,str,str,bool,i64,str,str
"""blibli""","""Buah Mangga Manalagi segar""","""1/2 kg""",true,500,"""variant""","""berat"""
"""blibli""","""MANGGA MANALAGI 170 - 180 G""","""""",false,175,"""nama""","""berat"""
"""blibli""","""MANGGA MANALAGI 550-570 GR""","""""",false,560,"""nama""","""berat"""
"""blibli""","""Mangga Manalagi""","""1 kg""",false,1000,"""variant""","""berat"""
"""blibli""","""Mangga Manalagi 1 kg ( 0.9 - 1.1 kg / pack)""","""""",false,1000,"""nama""","""berat"""
"""blibli""","""Mangga Manalagi 500 gram ( 400 - 500 gram / pack)""","""""",false,500,"""nama""","""berat"""
"""blibli""","""Mangga Manalagi Buah Segar Ibaza fruitsBOX - 1 kg Buah Mangga Manalagi""","""""",false,1000,"""nama""","""berat"""
"""blibli""","""Mangga manalagi segar manis 1/2 kg""","""""",false,500,"""nama""","""berat"""
"""tokopedia""","""1 Kg Mangga MANALAGI MADU Manis Legit dikirim Mengkal""","""https://www.tokopedia.com/murahmurahkediri3/1-kg-mangga-manalagi-madu-manis-legit-dikirim-…",false,1000,"""nama""","""berat"""


### Hasil per opsi: produk mangga Blibli yang punya varian (tanggal terbaru)

Satu nama, beberapa opsi (`item_sku`), harga berbeda. **Keputusan A1 (8 Nov):** berat/jumlah
hanya dari `variant` opsi itu, **tidak pernah dipinjam dari nama** ("Buy 1 Get 1" → `lain`). `harga_per_kg` = pratinjau Task 7.
Data opsi baru ada sejak 7–8 Nov.

In [31]:
opsi = []
for berkas in sorted(glob.glob("data_lake/raw/blibli/date=*/varian/ps--*.json")):
    ringkasan = json.load(open(berkas, encoding="utf-8"))["data"]
    if "mangga" not in (ringkasan.get("name") or "").lower():
        continue
    for o in ringkasan.get("options") or []:
        kode = o["id"].split("_")[0]                      # "is--SKU_PP-123" -> "is--SKU"
        if o.get("selected"):
            harga = ringkasan["price"]["offered"]         # opsi default: harga ada di ps--
        else:
            berkas_opsi = Path(berkas).parent / f"{kode}.json"
            harga = (json.load(open(berkas_opsi, encoding="utf-8"))["data"]["price"]["offered"]
                     if berkas_opsi.exists() else None)
        variant = o["attributes"][0]["value"] if o.get("attributes") else None
        s = parser.satuan_baris(ringkasan["name"], variant, banyak_opsi=True)
        opsi.append({
            "tanggal": berkas.split("date=")[1][:10],
            "nama": ringkasan["name"],
            "item_sku": kode.removeprefix("is--"),
            "opsi_default": bool(o.get("selected")),
            "variant": variant,
            "harga": harga,
            "berat_gram": s.berat_gram,
            "sumber_berat": s.sumber_berat,
            "jenis_satuan": s.jenis_satuan,
            "jumlah_buah": s.jumlah_buah,
            "harga_per_kg": round(harga / s.berat_gram * 1000) if harga and s.berat_gram else None,
            "harga_per_buah": round(harga / s.jumlah_buah) if harga and s.jumlah_buah else None,
        })

opsi = pl.DataFrame(opsi)
opsi.filter(pl.col("tanggal") == pl.col("tanggal").max()).sort("nama", "berat_gram")

tanggal,nama,item_sku,opsi_default,variant,harga,berat_gram,sumber_berat,jenis_satuan,jumlah_buah,harga_per_kg,harga_per_buah
str,str,str,bool,str,f64,i64,str,str,null,i64,null
"""2026-11-08""","""BUAH MANGGA HARUM MANIS SUPER PREMIUM QUALITY""","""BUS-70370-00021-00001""",true,"""800 GR""",30000.0,800,"""variant""","""berat""",null,37500,null
"""2026-11-08""","""BUAH MANGGA HARUM MANIS SUPER PREMIUM QUALITY""","""BUS-70370-00021-00002""",false,"""900 GR""",33000.0,900,"""variant""","""berat""",null,36667,null
"""2026-11-08""","""BUAH MANGGA HARUM MANIS SUPER PREMIUM QUALITY""","""BUS-70370-00021-00003""",false,"""1.000 GR""",36000.0,1000,"""variant""","""berat""",null,36000,null
"""2026-11-08""","""BUAH MANGGA HARUM MANIS SUPER PREMIUM QUALITY""","""BUS-70370-00021-00004""",false,"""PAKET 2 KG""",70000.0,2000,"""variant""","""berat""",null,35000,null
"""2026-11-08""","""Buah Mangga Indramayu""","""DOD-70072-00729-00002""",true,"""1/2 kg""",13500.0,500,"""variant""","""berat""",null,27000,null
"""2026-11-08""","""Buah Mangga Indramayu""","""DOD-70072-00729-00001""",false,"""1 kg""",20000.0,1000,"""variant""","""berat""",null,20000,null
"""2026-11-08""","""Buah Mangga Kweni""","""DOD-70072-00873-00002""",true,"""500g""",13000.0,500,"""variant""","""berat""",null,26000,null
"""2026-11-08""","""Buah Mangga Kweni""","""DOD-70072-00873-00001""",false,"""1000g""",25000.0,1000,"""variant""","""berat""",null,25000,null
"""2026-11-08""","""Buah Mangga Manalagi segar""","""DOD-70072-00076-00001""",true,"""1/2 kg""",13000.0,500,"""variant""","""berat""",null,26000,null


### Yang perlu ditinjau (3 pertanyaan, ±5 menit)

1. **Tanpa berat:** benarkah tidak ada berat tertulis di nama-nama ini?
2. **Nama sulit** (rentang, pecahan, garis miring): benarkah `berat_gram`-nya?
3. **Opsi yang mencurigakan:** opsi yang `harga_per_kg`-nya beda > 30% dari opsi lain di produk yang sama
   (tanda berat salah). Tabel kosong = tidak ada yang mencurigakan.

In [32]:
display(berat.filter(pl.col("jenis_satuan") != "berat").select("sumber", "nama"))
display(berat.filter(pl.col("nama").str.contains(r"\d\s*-\s*\d|/")).select("sumber", "nama", "berat_gram"))
terbaru = opsi.filter(pl.col("tanggal") == pl.col("tanggal").max())
terbaru.with_columns(median_produk=pl.col("harga_per_kg").median().over("nama")).filter(
    (pl.col("harga_per_kg") / pl.col("median_produk") - 1).abs() > 0.3
).select("nama", "variant", "harga", "berat_gram", "harga_per_kg", "median_produk")

sumber,nama
str,str
"""tokopedia""","""MANGGA MANALAGI MADU MANGGA MANALAGI MATANG MANIS Buah"""
"""tokopedia""","""Mangga Madu Manalagi Fresh - Buah Segar dengan Rasa Manis dan Nikmat"""
"""tokopedia""","""Mangga Manalagi Potong Manis"""
"""tokopedia""","""Mangga Manalagi Potong Manis"""
"""tokopedia""","""mangga manalagi jumbo isi 2 sampai 3 mentah tua rasa manis mantap"""
"""tokopedia""","""mangga manalagi jumbo isi 2 sampai 3 mentah tua rasa manis mantap"""
"""tokopedia""","""mangga manalagi mentah tua rasa manis dam mantap"""


sumber,nama,berat_gram
str,str,i64
"""blibli""","""MANGGA MANALAGI 170 - 180 G""",175
"""blibli""","""MANGGA MANALAGI 550-570 GR""",560
"""blibli""","""Mangga Manalagi 1 kg ( 0.9 - 1.1 kg / pack)""",1000
"""blibli""","""Mangga Manalagi 500 gram ( 400 - 500 gram / pack)""",500
"""blibli""","""Mangga manalagi segar manis 1/2 kg""",500
"""tokopedia""","""Mangga Madu keraton/Manalagi muda 1kg""",1000
"""tokopedia""","""Mangga Manalagi /1 kg""",1000
"""tokopedia""","""Mangga Manalagi Madu (Arum ) Tua Pohon asli Situbondo, isi 3-5/ kg - Buah""",1000
"""tokopedia""","""Mangga manalagi madu (Arum )fres & segar tua pohon 1kg isi 3-5 buah.""",1000


nama,variant,harga,berat_gram,harga_per_kg,median_produk
str,str,f64,i64,i64,f64


### Coba nama sendiri

Ganti teks di bawah lalu jalankan. Produk **tanpa opsi**: kosongkan `contoh_variant` (`""`).

In [33]:
contoh_nama = "Mangga Manalagi 900g - 1 Kg"
contoh_variant = "1 kg"   # "" kalau produk tanpa opsi

berat_contoh, sumber_contoh = parser.pilih_berat([("variant", contoh_variant), ("nama", contoh_nama)])
berat_contoh, sumber_contoh, parser.jenis_satuan(contoh_nama, berat_contoh)

(1000, 'variant', 'berat')

## Task 1B: jenis buah, olahan, asal, terjual, region (`transform/parser.py`)

- `jenis_buah(nama)`: kata kunci → `mangga`, `apel`, …, `lainnya`. Mangga ikut aturan salah ketik.
- `olahan(nama)`: penanda produk olahan (tanpa `sale` dan `pohon`, keputusan 8 Nov).
- `asal_dari_nama(nama)`: `impor` / `lokal` / `tidak_diketahui` (untuk Tokopedia).
- `terjual_dari_label(label)`, `ember_terjual(n)`: angka terjual Tokopedia / Blibli disetarakan.
- `region(kota)`: `jabodetabek` / `jawa_lain` / `luar_jawa` / `tidak_diketahui`.
- Multipack (`berat_dikali`): "8x125GR" → 1000 g, dicatat di karantina nanti (cek H).

### Kode

In [34]:
sumber = inspect.getsource(parser)
tampilkan_kode(sumber[sumber.index("# ---- Task 1B ----"):])

### Hasil (tanggal terbaru, Blibli + Tokopedia)

In [35]:
tanggal_baru = max(Path(f).name for f in glob.glob("data_lake/raw/blibli/date=*"))
produk = []
for berkas in glob.glob(f"data_lake/raw/blibli/{tanggal_baru}/*_page_*.json"):
    label_asal = "lokal" if "BU-1000190" in berkas else "impor"
    for p in (json.load(open(berkas, encoding="utf-8")).get("data") or {}).get("products") or []:
        produk.append({"sumber": "blibli", "nama": p["name"], "kota": p.get("location"),
                       "label_asal": label_asal, "label_terjual": None})
for berkas in glob.glob(f"data_lake/raw/tokopedia/{tanggal_baru}/page_*.json"):
    try:
        isi = json.load(open(berkas, encoding="utf-8"))[0]["data"]["searchProductV5"]["data"]["products"]
    except (ValueError, KeyError, IndexError, TypeError):
        continue
    for p in isi:
        label = next((l.get("title") for l in p.get("labelGroups") or []
                      if l.get("position") == "ri_product_credibility"), None)
        produk.append({"sumber": "tokopedia", "nama": p["name"], "kota": p["shop"].get("city"),
                       "label_asal": None, "label_terjual": label})

lain = pl.DataFrame(produk, infer_schema_length=None).with_columns(
    jenis_buah=pl.col("nama").map_elements(parser.jenis_buah, return_dtype=pl.Utf8),
    olahan=pl.col("nama").map_elements(parser.olahan, return_dtype=pl.Boolean),
    asal=pl.col("nama").map_elements(parser.asal_dari_nama, return_dtype=pl.Utf8),
    region=pl.col("kota").map_elements(parser.region, return_dtype=pl.Utf8, skip_nulls=False),
    terjual=pl.col("label_terjual").map_elements(parser.terjual_dari_label, return_dtype=pl.Int64),
    berat_gram=pl.col("nama").map_elements(parser.berat_gram, return_dtype=pl.Int64),
    berat_dikali=pl.col("nama").map_elements(parser.berat_dikali, return_dtype=pl.Boolean),
)
print(tanggal_baru, lain.height, "baris")
display(lain.group_by("jenis_buah").len().sort("len", descending=True))
display(lain.group_by("region").len().sort("len", descending=True))

date=2026-11-08 7600 baris


jenis_buah,len
str,u32
"""lainnya""",2811
"""jeruk""",1151
"""apel""",514
"""mangga""",428
"""anggur""",420
"""pir""",383
"""pisang""",331
"""jambu""",219
"""alpukat""",195


region,len
str,u32
"""jabodetabek""",5528
"""jawa_lain""",1733
"""luar_jawa""",336
"""tidak_diketahui""",3


### Yang perlu ditinjau

1. **Olahan:** apakah baris-baris ini memang bukan buah utuh?
2. **Asal (Blibli punya label asli):** baris yang aturannya salah.
3. **Multipack:** apakah hasil perkaliannya benar?
4. **Kota luar Jawa:** adakah kota Jawa yang salah masuk ke sini?

In [36]:
display(lain.filter(pl.col("olahan")).select("sumber", "nama").unique().sort("nama").head(40))
display(lain.filter(pl.col("label_asal").is_not_null() & (pl.col("asal") != "tidak_diketahui")
                    & (pl.col("asal") != pl.col("label_asal"))).select("nama", "label_asal", "asal").unique())
display(lain.filter(pl.col("berat_dikali")).select("sumber", "nama", "berat_gram").unique())
lain.filter(pl.col("region") == "luar_jawa").select("kota").unique().sort("kota")

sumber,nama
str,str
"""tokopedia""","""(1000 GRAM) MANISAN BASAH ANEKA BUAH ASLI CIANJUR ANGGUR BOGOR SALAK MANGGA KEDONDONG BANG…"
"""tokopedia""","""1 KG MANISAN BUAH KHAS CIANJUR MANGGA SALAK PEPAYA CANAR KEDONGDONG LOKAL KEDONGDONG BANGK…"
"""tokopedia""","""1 KG MANISAN MANGGA BASAH / ASINAN BUAH MANGA AIR SEGAR / FRESH MANGO"""
"""tokopedia""","""1 Kg Buah Pinang Tua Segar Pilihan Terbaik Isi Sekitar 15-30 Biji | Buah Pinang Muda Fress…"
"""tokopedia""","""100 gr Dried Figs / Dry Fig / Buah Tin / Buah Tin Ara Kering Turkey"""
"""tokopedia""","""100 gr Dried Papaya / Buah Pepaya Kering / Dry Pepaya 100gr 250gr"""
"""blibli""","""1000gr Kichi / Dried Goji Berry"""
"""blibli""","""1000gr Kranberi Merah Kering / Dried Red Cranberry"""
"""tokopedia""","""100gr Gojiberi Kering - Dried Gojiberry - Kichi"""


nama,label_asal,asal
str,str,str
"""Jeruk Sunkist Cara Cara (0.9 - 1.1 kg / pack)""","""lokal""","""impor"""
"""Kepok Kuning""","""impor""","""lokal"""
"""Apel Fuji Blush ( 500 - 650 gram / pack)""","""lokal""","""impor"""
"""KIWI RED PUNNET 6 PCS""","""lokal""","""impor"""
"""My Fruit Surabaya Jeruk Santang Lokal Buah [800gr]""","""impor""","""lokal"""
"""ANGGUR SHINE MUSCAT 450GR""","""lokal""","""impor"""
"""Jeruk Santang Lokal 1 Kg segar TOP Pilihan""","""impor""","""lokal"""
"""Blueberry Impor (125 gram / pack)""","""lokal""","""impor"""
"""Buah Segar Jeruk Sunkist Navel RRC FRESHGO""","""lokal""","""impor"""


sumber,nama,berat_gram
str,str,i64
"""blibli""","""BLUEBERRY TRAIN TOY 8x125GR PACK""",1000
"""tokopedia""","""Jeruk Shantang Yuji Royal 500 Grm X 10 Pack""",5000
"""tokopedia""","""Kurma Akram Bam Mazafati 550g x 12 (1 dus)""",6600
"""tokopedia""","""Jeruk Shantang Daun Jaring Manis Isi 8 X 700 Grm / Box""",5600


kota
str
"""Balikpapan"""
"""Bandar Lampung"""
"""Banjarbaru"""
"""Banjarmasin"""
"""Batam"""
"""Binjai"""
"""Bukittinggi"""
"""Denpasar"""
"""Kab. Badung"""


## Task 4: transform Blibli (`transform/blibli.py`)

Semua berkas mentah Blibli satu tanggal → tabel `blibli_produk`, **satu baris per opsi** (`item_sku`).
- baris default dari kartu hasil cari, baris opsi lain dari `varian/is--*.json`;
- kolom produk (penjual, kategori, rating) disalin ke baris opsi; kolom khusus kartu (voucher, kampanye) NULL;
- opsi tanpa harga (tidak tersedia) tetap ada dengan `harga` NULL (keputusan B).

### Kode

In [37]:
from transform import blibli
importlib.reload(blibli)
tampilkan_kode(blibli)

### Hasil: semua tanggal

In [38]:
from datetime import date

blibli_produk = pl.concat([
    blibli.ubah(folder, date.fromisoformat(folder.name.removeprefix("date=")))
    for folder in sorted(Path("data_lake/raw/blibli").glob("date=*"))
])
print(blibli_produk.height, "baris,", blibli_produk.width, "kolom")
blibli_produk.group_by("tanggal").agg(
    baris=pl.len(),
    opsi_lain=(~pl.col("opsi_default")).sum(),
    harga_null=pl.col("harga").is_null().sum(),
    berat_terbaca=pl.col("berat_gram").is_not_null().sum(),
).sort("tanggal")

24648 baris, 34 kolom


tanggal,baris,opsi_lain,harga_null,berat_terbaca
date,u32,u32,u32,u32
2026-10-18,1600,0,0,1060
2026-10-19,1600,0,0,1076
2026-10-27,1600,0,0,1066
2026-10-28,1600,0,0,1065
2026-10-29,1600,0,0,1068
2026-10-30,1599,0,0,1070
2026-10-31,1600,0,0,1068
2026-11-01,1599,0,0,1063
2026-11-02,1600,0,0,1071


### Yang perlu ditinjau

1. **Mangga Manalagi, tanggal terbaru:** satu baris per opsi, harga dan berat benar?
2. **Opsi tanpa harga:** benarkah ini opsi yang tidak tersedia?
3. **Satu produk lengkap** (semua 34 kolom) untuk dicek dengan halaman Blibli.

In [39]:
terbaru = blibli_produk.filter(pl.col("tanggal") == pl.col("tanggal").max())
display(terbaru.filter(pl.col("nama").map_elements(parser.mangga_manalagi, return_dtype=pl.Boolean))
        .select("item_sku", "opsi_default", "nama", "variant", "harga", "harga_coret", "berat_gram",
                "sumber_berat", "penjual", "url"))
display(terbaru.filter(pl.col("harga").is_null()).select("item_sku", "nama", "variant", "penjual").head(15))
terbaru.filter(pl.col("produk_id") == "DOD-70072-00076").transpose(include_header=True)

item_sku,opsi_default,nama,variant,harga,harga_coret,berat_gram,sumber_berat,penjual,url
str,bool,str,str,i64,i64,i64,str,str,str
"""RAM-70107-29314-00001""",true,"""MANGGA MANALAGI 170 - 180 G""",null,6800,6800,175,"""nama""","""Ranch Market Flagship Store""","""https://www.blibli.com/p/mangga-manalagi-170-180-g/ps--RAM-70107-29314?ds=RAM-70107-29314-…"
"""PTS-70170-01898-00001""",true,"""Mangga Manalagi 1 kg ( 0.9 - 1.1 kg / pack)""",null,14000,14000,1000,"""nama""","""BlibliFresh""","""https://www.blibli.com/p/mangga-manalagi-1-kg-0-9-1-1-kg-pack/ps--PTS-70170-01898?ds=PTS-7…"
"""SES-70872-07089-00001""",true,"""Mangga Manalagi 500 gram ( 400 - 500 gram / pack)""",null,6400,6800,500,"""nama""","""Segari Supermarket Flagship Store""","""https://www.blibli.com/p/mangga-manalagi-500-gram-400-500-gram-pack/ps--SES-70872-07089?ds…"
"""PAB-70112-01523-00001""",true,"""Mangga manalagi segar manis 1/2 kg""",null,26865,27000,500,"""nama""","""Pasar badung""","""https://www.blibli.com/p/mangga-manalagi-segar-manis-1-2-kg/ps--PAB-70112-01523?ds=PAB-701…"
"""DOD-70072-00076-00001""",true,"""Buah Mangga Manalagi segar""","""1/2 kg""",13000,13000,500,"""variant""","""Doyan Daging Store Depok""","""https://www.blibli.com/p/buah-mangga-manalagi-segar/ps--DOD-70072-00076?ds=DOD-70072-00076…"
"""SAX-70009-02879-00001""",true,"""Mangga Manalagi""","""1 kg""",25700,25700,1000,"""variant""","""Sayurbox Flagship Store""","""https://www.blibli.com/p/mangga-manalagi/ps--SAX-70009-02879?ds=SAX-70009-02879-00001"""
"""DOD-70072-00076-00002""",false,"""Buah Mangga Manalagi segar""","""1 kg""",25000,25000,1000,"""variant""","""Doyan Daging Store Depok""","""https://www.blibli.com/p/buah-mangga-manalagi-segar/ps--DOD-70072-00076?ds=DOD-70072-00076…"


item_sku,nama,variant,penjual
str,str,str,str
"""FRI-70030-00021-00003""","""Buah Segar Anggur Autumn USA FRESHGO""","""1kg""","""FreshGO Indonesia"""
"""FRI-70030-00241-00002""","""Anggur Shine Muscat import | FRESHGO""","""1/4 kg""","""FreshGO Indonesia"""
"""HAM-70080-00993-00010""","""HasilSumberalaM Anggur Import Manis Segar Shine Muscat Autumn Royal Red Globe Crimson Cand…","""Candy Snaps Permen 250gr""","""HasilSumberalaM"""
"""HAM-70080-00993-00011""","""HasilSumberalaM Anggur Import Manis Segar Shine Muscat Autumn Royal Red Globe Crimson Cand…","""Candy Snaps Permen 500gr""","""HasilSumberalaM"""
"""HAM-70080-00993-00012""","""HasilSumberalaM Anggur Import Manis Segar Shine Muscat Autumn Royal Red Globe Crimson Cand…","""Candy Snaps Permen 1kg""","""HasilSumberalaM"""
"""HAM-70126-00017-00005""","""HarvestMoon Nanas Honi Sunpride Per Pcs""","""Potong""","""Harvest Moon Jakarta"""
"""HAM-70126-00596-00002""","""HarvestMoon Kiwi Gold Zespri New Zealand per Kg""","""Jumbo""","""Harvest Moon Jakarta"""
"""IBF-60022-00179-00001""","""Buah Jeruk Impor Sunkist Lemon Santang Ponkan Wogan Murcot Mandarin :""","""Jeruk Afourer""","""IBAZA fruitsBOX"""
"""IBF-60022-00179-00002""","""Buah Jeruk Impor Sunkist Lemon Santang Ponkan Wogan Murcot Mandarin :""","""Jeruk Kino""","""IBAZA fruitsBOX"""


column,column_0,column_1
str,str,str
"""tanggal""","""2026-11-08""","""2026-11-08"""
"""produk_id""","""DOD-70072-00076""","""DOD-70072-00076"""
"""item_sku""","""DOD-70072-00076-00001""","""DOD-70072-00076-00002"""
"""opsi_default""","""true""","""false"""
"""nama""","""Buah Mangga Manalagi segar""","""Buah Mangga Manalagi segar"""
"""variant""","""1/2 kg""","""1 kg"""
"""harga""","""13000""","""25000"""
"""harga_coret""","""13000""","""25000"""
"""diskon_persen""","""0""","""0"""


In [40]:
terbaru.head()

tanggal,produk_id,item_sku,opsi_default,nama,variant,harga,harga_coret,diskon_persen,harga_voucher,harga_coret_voucher,diskon_voucher_persen,kampanye,harga_sebelum_kampanye,merchant_code,penjual,resmi,lokasi,lencana_toko,rating_toko,merek,kategori,kategori_tarik,rating,jumlah_ulasan,total_terjual,terjual_bulan_ini_min,url,sku,berat_gram,sumber_berat,jenis_satuan,jumlah_buah,berat_dikali
date,str,str,bool,str,str,i64,i64,i64,i64,i64,i64,str,i64,str,str,bool,str,str,f64,str,str,str,f64,i64,i64,i64,str,str,i64,str,str,i64,bool
2026-11-08,"""HYP-60026-02571""","""HYP-60026-02571-00001""",true,"""PEPAYA CALIFORNIA 1300 - 1500 GR""",null,20850,20850,0,null,null,null,null,20850,"""HYP-60026""","""Hypermart Flagship Store""",true,"""Kota Surabaya""","""Diamond""",4.8,"""Hypermart""","""Bliblimart > Sayuran & Buah - buahan > Buah - buahan > Buah Lokal""","""BU-1000190""",4.7,882,6956,50,"""https://www.blibli.com/p/pepaya-california-1300-1500-gr/ps--HYP-60026-02571?ds=HYP-60026-0…","""HYP-60026-02571""",1400,"""nama""","""berat""",null,false
2026-11-08,"""RAM-70107-10697""","""RAM-70107-10697-00001""",true,"""DRAGON FRUIT RED""",null,38400,38400,0,null,null,null,null,38400,"""RAM-70107""","""Ranch Market Flagship Store""",true,"""Kota Surabaya""","""Gold""",4.8,"""Ranch Market""","""Bliblimart > Sayuran & Buah - buahan > Buah - buahan > Buah Lokal""","""BU-1000190""",4.8,148,1544,null,"""https://www.blibli.com/p/dragon-fruit-red/ps--RAM-70107-10697?ds=RAM-70107-10697-00001""","""RAM-70107-10697""",null,null,"""lain""",null,false
2026-11-08,"""HYP-60026-23815""","""HYP-60026-23815-00001""",true,"""JERUK CARACARA 540 - 600 GR""",null,24300,24300,0,null,null,null,null,24300,"""HYP-60026""","""Hypermart Flagship Store""",true,"""Kota Surabaya""","""Diamond""",4.8,"""Hypermart""","""Bliblimart > Sayuran & Buah - buahan > Buah - buahan > Buah Lokal""","""BU-1000190""",5.0,2,13,null,"""https://www.blibli.com/p/jeruk-caracara-540-600-gr/ps--HYP-60026-23815?ds=HYP-60026-23815-…","""HYP-60026-23815""",570,"""nama""","""berat""",null,false
2026-11-08,"""RAM-70107-29314""","""RAM-70107-29314-00001""",true,"""MANGGA MANALAGI 170 - 180 G""",null,6800,6800,0,null,null,null,null,6800,"""RAM-70107""","""Ranch Market Flagship Store""",true,"""Kota Surabaya""","""Gold""",4.8,"""no brand""","""Bliblimart > Sayuran & Buah - buahan > Buah - buahan > Buah Lokal""","""BU-1000190""",4.2,9,157,null,"""https://www.blibli.com/p/mangga-manalagi-170-180-g/ps--RAM-70107-29314?ds=RAM-70107-29314-…","""RAM-70107-29314""",175,"""nama""","""berat""",null,false
2026-11-08,"""HYP-60026-23829""","""HYP-60026-23829-00001""",true,"""JERUK CLEMENGOLD 540 - 600 GR""",null,53940,53940,0,null,null,null,null,53940,"""HYP-60026""","""Hypermart Flagship Store""",true,"""Kota Surabaya""","""Diamond""",4.8,"""Hypermart""","""Bliblimart > Sayuran & Buah - buahan > Buah - buahan > Buah Lokal""","""BU-1000190""",1.0,1,11,null,"""https://www.blibli.com/p/jeruk-clemengold-540-600-gr/ps--HYP-60026-23829?ds=HYP-60026-2382…","""HYP-60026-23829""",570,"""nama""","""berat""",null,false


## Task 5 (sebelum kode): tinjau harga coret Tokopedia (pertanyaan 1)

Tiap kartu hasil cari Tokopedia punya `price = {number, original, discountPercentage}`:
- `harga` ← `number` (harga bayar)
- `harga_asli` ← `original` (harga coret, sering kosong)
- `diskon_tokopedia` ← `discountPercentage` (selalu 0?)
- `diskon_hitung` = (harga_asli − harga) / harga_asli × 100

Tabel di bawah **bukan** hasil transform; hanya kolom harga dari data mentah, satu baris per (tanggal, produk).

In [41]:
from datetime import date
baris = []
for folder in sorted(Path("data_lake/raw/tokopedia").glob("date=*")):
    tanggal = date.fromisoformat(folder.name.removeprefix("date="))
    for berkas in sorted(folder.glob("page_*.json")):
        isi = json.load(open(berkas, encoding="utf-8"))
        if isinstance(isi, list):
            isi = isi[0]
        for p in isi["data"]["searchProductV5"]["data"]["products"]:
            harga = p.get("price") or {}
            baris.append({
                "tanggal": tanggal,
                "produk_id": str(p["id"]),
                "nama": p["name"],
                "toko": (p.get("shop") or {}).get("name"),
                "harga": harga.get("number"),
                "harga_asli": int("".join(c for c in harga.get("original") or "" if c.isdigit()) or 0) or None,
                "diskon_tokopedia": harga.get("discountPercentage"),
                "url": p["url"].split("?")[0],
            })
toko_harga = (
    pl.DataFrame(baris)
    .unique(["tanggal", "produk_id"], keep="first", maintain_order=True)
    .with_columns(
        selisih=pl.col("harga_asli") - pl.col("harga"),
        diskon_hitung=((pl.col("harga_asli") - pl.col("harga")) * 100 / pl.col("harga_asli")).round(1),
        manalagi=pl.col("nama").map_elements(parser.mangga_manalagi, return_dtype=pl.Boolean),
    )
)
print(toko_harga.height, "baris")
toko_harga.group_by("tanggal").agg(
    baris=pl.len(),
    ada_harga_asli=pl.col("harga_asli").is_not_null().sum(),
    diskon_tokopedia_bukan_0=(pl.col("diskon_tokopedia") != 0).sum(),
).sort("tanggal")

77799 baris


tanggal,baris,ada_harga_asli,diskon_tokopedia_bukan_0
date,u32,u32,u32
2026-10-23,5745,1784,0
2026-10-27,5669,1173,0
2026-10-28,5666,1174,0
2026-10-29,5625,1098,0
2026-10-30,4725,1041,0
2026-10-31,5635,1087,0
2026-11-01,5599,1080,0
2026-11-02,5532,1061,0
2026-11-03,5676,1094,0


### Per produk: dicoret selalu, kadang, atau tidak pernah?

- **selalu dicoret**: harga coret tiap hari → curiga diskon palsu (cek halaman produknya).
- **kadang dicoret**: harga coret hanya beberapa hari → kemungkinan diskon sungguhan.

Ganti `hanya_manalagi = False` untuk melihat semua produk.

In [42]:
hanya_manalagi = True   # False = semua produk Tokopedia

per_produk = (
    toko_harga.filter(pl.col("manalagi") | (not hanya_manalagi))
    .group_by("produk_id")
    .agg(
        nama=pl.col("nama").first(),
        toko=pl.col("toko").first(),
        hari_muncul=pl.len(),
        hari_dicoret=pl.col("harga_asli").is_not_null().sum(),
        harga_min=pl.col("harga").min(),
        harga_max=pl.col("harga").max(),
        diskon_hitung_max=pl.col("diskon_hitung").max(),
        url=pl.col("url").first(),
    )
    .with_columns(
        pola=pl.when(pl.col("hari_dicoret") == 0).then(pl.lit("tidak pernah dicoret"))
        .when(pl.col("hari_dicoret") == pl.col("hari_muncul")).then(pl.lit("selalu dicoret"))
        .otherwise(pl.lit("kadang dicoret"))
    )
    .sort("pola", "hari_dicoret", descending=[False, True])
)
display(per_produk.group_by("pola").agg(produk=pl.len()).sort("pola"))
per_produk

pola,produk
str,u32
"""selalu dicoret""",2
"""tidak pernah dicoret""",39


produk_id,nama,toko,hari_muncul,hari_dicoret,harga_min,harga_max,diskon_hitung_max,url,pola
str,str,str,u32,u32,i64,i64,f64,str,str
"""8061245026""","""MANGGA MANALAGI MADU MANGGA MANALAGI MATANG MANIS Buah""","""Mantra Khatulistiwa""",14,14,14550,29100,3.0,"""https://www.tokopedia.com/mantra-khatulistiwa/mangga-manalagi-madu-mangga-manalagi-matang-…","""selalu dicoret"""
"""1256380209""","""Mangga Manalagi 500 Gr Fresh Murah""","""Aneka Sayuran Segar""",14,14,24750,24750,1.0,"""https://www.tokopedia.com/sayuransegarjkt/mangga-manalagi-500-gr-fresh-murah""","""selalu dicoret"""
"""103597964651""","""mangga manalagi mentah tua rasa manis dam mantap""","""ASIZEHSHOP 4""",7,0,12999,13999,null,"""https://www.tokopedia.com/asizehshop-4/mangga-manalagi-tua-pohon-rasa-manis-dam-mantap-173…","""tidak pernah dicoret"""
"""103404321784""","""Mangga manalagi, mangga manalagi fresh , mangga manalagi 1kg""","""mitra petani surabaya""",4,0,30000,30000,null,"""https://www.tokopedia.com/mitrapetanisurabaya/mangga-manalagi-mangga-manalagi-fresh-mangga…","""tidak pernah dicoret"""
"""102383650601""","""Mangga Manalagi Segar 1Kg – Manis dan Daging Tebal""","""Mracang Market""",2,0,14000,14000,null,"""https://www.tokopedia.com/mracangmarket/mangga-manalagi-segar-1kg-manis-dan-daging-tebal-1…","""tidak pernah dicoret"""
"""104001505646""","""1 Kg Mangga MANALAGI MADU Manis Legit dikirim Mengkal""","""MURAHMURAH.KEDIRI3""",1,0,15999,15999,null,"""https://www.tokopedia.com/murahmurahkediri3/1-kg-mangga-manalagi-madu-manis-legit-dikirim-…","""tidak pernah dicoret"""
"""9640686515""","""mangga manalagi madu 1 kg buah mangga manalagi""","""allfreshbuah02""",6,0,38000,38000,null,"""https://www.tokopedia.com/allfreshbuah02/mangga-manalagi-madu-1-kg-buah-mangga-manalagi""","""tidak pernah dicoret"""
"""103999725487""","""Mangga manalagi madu (Arum )fres & segar tua pohon 1kg isi 3-5 buah.""","""TOKO MANGGA SITUBONDO.2""",3,0,18999,18999,null,"""https://www.tokopedia.com/toko-mangga-situbondo2/mangga-manalagi-fres-segar-17356323591012…","""tidak pernah dicoret"""
"""14020755371""","""Mangga Manalagi Potong Manis""","""Siti Fresh""",7,0,10000,10000,null,"""https://www.tokopedia.com/sitifresh/mangga-manalagi-potong-manis""","""tidak pernah dicoret"""


### Satu produk dari hari ke hari

Ambil `produk_id` dari tabel di atas, lalu bandingkan dengan halaman produknya (`url`).

In [43]:
contoh_id = "7286804348"   # ganti dengan produk_id dari tabel di atas
toko_harga.filter(pl.col("produk_id") == contoh_id).select(
    "tanggal", "harga", "harga_asli", "selisih", "diskon_hitung", "diskon_tokopedia"
).sort("tanggal")

tanggal,harga,harga_asli,selisih,diskon_hitung,diskon_tokopedia
date,i64,i64,i64,f64,i64
2026-10-23,250800,285000,34200,12.0,0
2026-10-27,285000,null,null,null,0
2026-10-28,285000,null,null,null,0
2026-10-29,285000,null,null,null,0
2026-10-30,285000,null,null,null,0
2026-10-31,285000,null,null,null,0
2026-11-01,285000,null,null,null,0
2026-11-02,285000,null,null,null,0
2026-11-04,285000,null,null,null,0


## Task 5: transform Tokopedia (`transform/tokopedia.py`)

Semua berkas mentah Tokopedia satu tanggal → tabel `tokopedia_produk`, **satu baris per opsi** (sama seperti Blibli, spec 5.0):
- baris kartu (`opsi_default = True`) dari hasil cari;
- baris opsi lain dari halaman produk `halaman/<id>.json` (hanya Mangga Manalagi, mulai 8 Nov);
- berat: **opsi → halaman (hanya kalau nama tanpa berat) → nama → URL**;
- `harga_asli` disimpan tapi tidak dipakai di fakta, `diskon_persen` selalu NULL; hanya salinan laptop.

### Kode

In [ ]:
from transform import tokopedia
importlib.reload(tokopedia)
tampilkan_kode(tokopedia)

### Hasil: semua tanggal

In [ ]:
from datetime import date

tokopedia_produk = pl.concat([
    tokopedia.ubah(folder, date.fromisoformat(folder.name.removeprefix("date=")))
    for folder in sorted(Path("data_lake/raw/tokopedia").glob("date=*"))
])
print(tokopedia_produk.height, "baris,", tokopedia_produk.width, "kolom")
tokopedia_produk.group_by("tanggal").agg(
    baris=pl.len(),
    opsi_lain=(~pl.col("opsi_default")).sum(),
    berat_terbaca=pl.col("berat_gram").is_not_null().sum(),
    kota_null=pl.col("kota").is_null().sum(),
).sort("tanggal")

### Yang perlu ditinjau

1. **Mangga Manalagi, tanggal terbaru**, urut harga per kg: berat dan `sumber_berat` masuk akal? Cocokkan beberapa dengan halaman produknya (`url`).
2. **Satu produk bervarian lengkap**: baris kartu + semua opsinya.

In [ ]:
terbaru_tp = tokopedia_produk.filter(pl.col("tanggal") == pl.col("tanggal").max())
manalagi_tp = (
    terbaru_tp.filter(pl.col("nama").map_elements(parser.mangga_manalagi, return_dtype=pl.Boolean))
    .with_columns(harga_per_kg=(pl.col("harga") * 1000 / pl.col("berat_gram")).round())
)
display(manalagi_tp.group_by("sumber_berat").agg(baris=pl.len()))
display(manalagi_tp.select("produk_id", "opsi_default", "nama", "variant", "harga", "berat_gram",
                           "sumber_berat", "harga_per_kg", "nama_toko", "url").sort("harga_per_kg"))
terbaru_tp.filter(pl.col("induk_id") == "103388173219").select(
    "produk_id", "opsi_default", "variant", "harga", "berat_gram", "sumber_berat", "url")